In [4]:
"""
“The movie was full of”に続くトークン(トークン列ではなく一つのトークンであることに注意せよ)
として適切なもの上位10個と、その確率(尤度)を求めよ。
ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。
"""
from transformers import AutoTokenizer,AutoModelForCausalLM,GPT2LMHeadModel
import torch
import torch.nn.functional as F

# トークナイザーの読み込み
tokenizer=AutoTokenizer.from_pretrained("openai-community/gpt2-medium")
tokenizer.pad_token=tokenizer.eos_token # GPT2に必要
# モデルの読み込み
#   CausalLM：因果言語モデリングに特化したもの。主に文章の続きを予測するタスクに使われる。
model=AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium")
model.eval()

# プロンプトの設定
prompt="The movie was full of"
inputs=tokenizer(prompt,return_tensors="pt")
input_ids=inputs["input_ids"]

# トークン列確認
#   -.convert_ids_to_tokensによりトークンIDを可読なトークン列に戻す
print("トークン列:",tokenizer.convert_ids_to_tokens(input_ids[0]))

# 次のトークンの確率を計算
with torch.no_grad():
    outputs=model(**inputs)
next_token_logits=outputs.logits[0,-1,:]
probs=F.softmax(next_token_logits,dim=-1)

# 上位10トークンと確率を取得
top_k=10
#   -最大確率上位k個のトークンとそのIDを取得
top_probs,top_indices=torch.topk(probs,k=top_k)
top_tokens=tokenizer.convert_ids_to_tokens(top_indices)

# 結果の出力
for i in range(top_k):
    print(f"{i+1}: Token = {top_tokens[i]}, Probability = {top_probs[i].item():.6f}")
#実行結果

トークン列: ['The', 'Ġmovie', 'Ġwas', 'Ġfull', 'Ġof']
1: Token = Ġgreat, Probability = 0.023094
2: Token = Ġreferences, Probability = 0.013512
3: Token = Ġaction, Probability = 0.013043
4: Token = Ġmoments, Probability = 0.012450
5: Token = Ġthe, Probability = 0.011860
6: Token = Ġcharacters, Probability = 0.008720
7: Token = Ġthese, Probability = 0.007216
8: Token = Ġsurprises, Probability = 0.006894
9: Token = Ġfun, Probability = 0.006526
10: Token = Ġthem, Probability = 0.006154
